# Patrones de Turing (Reaction-Diffusion) - Nivel intermedio

Objetivo:
- Implementar el modelo Gray-Scott.
- Explorar parametros y presets.
- Mostrar un preview en el notebook.
- Integrar con ascii_stream_engine (source sintetico).

Notas:
- Usa solo numpy y pillow para visualizar.
- Para streaming via UDP, necesitas ffmpeg y VLC.


In [ ]:
import os
import sys
import time

import numpy as np
from PIL import Image
from IPython.display import display, clear_output

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

np.random.seed(42)


## Modelo Gray-Scott

Ecuaciones (discretas):

U_t = Du * Lap(U) - U*V*V + F*(1-U)

V_t = Dv * Lap(V) + U*V*V - (F + k) * V

Parametros:
- Du, Dv: diffusion rates
- F: feed rate
- k: kill rate
- dt: paso temporal

Tip: cambios pequeños en F y k producen patrones muy distintos.


In [ ]:
def laplacian(x: np.ndarray) -> np.ndarray:
    return (
        -x
        + 0.2 * (
            np.roll(x, 1, 0)
            + np.roll(x, -1, 0)
            + np.roll(x, 1, 1)
            + np.roll(x, -1, 1)
        )
        + 0.05 * (
            np.roll(x, (1, 1), (0, 1))
            + np.roll(x, (1, -1), (0, 1))
            + np.roll(x, (-1, 1), (0, 1))
            + np.roll(x, (-1, -1), (0, 1))
        )
    )


def init_fields(n: int, seed_size: int = 12, noise: float = 0.0):
    U = np.ones((n, n), dtype=np.float32)
    V = np.zeros((n, n), dtype=np.float32)
    s = n // 2
    r = seed_size // 2
    U[s - r : s + r, s - r : s + r] = 0.0
    V[s - r : s + r, s - r : s + r] = 1.0
    if noise > 0:
        U += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        V += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        np.clip(U, 0.0, 1.0, out=U)
        np.clip(V, 0.0, 1.0, out=V)
    return U, V


def gs_step(U, V, Du, Dv, F, k, dt):
    lapU = laplacian(U)
    lapV = laplacian(V)
    uvv = U * V * V
    U += (Du * lapU - uvv + F * (1 - U)) * dt
    V += (Dv * lapV + uvv - (F + k) * V) * dt
    np.clip(U, 0.0, 1.0, out=U)
    np.clip(V, 0.0, 1.0, out=V)
    return U, V


def render_gray(V: np.ndarray, size: int = 400) -> Image.Image:
    img = (V * 255).astype(np.uint8)
    return Image.fromarray(img, mode="L").resize((size, size))


In [ ]:
n = 200
U, V = init_fields(n, seed_size=12, noise=0.02)

Du, Dv = 0.16, 0.08
F, k = 0.035, 0.065
dt = 1.0

steps = 600
show_every = 30

for i in range(steps):
    U, V = gs_step(U, V, Du, Dv, F, k, dt)
    if i % show_every == 0:
        clear_output(wait=True)
        display(render_gray(V, size=400))
        print(f"step={i} F={F} k={k}")
        time.sleep(0.01)


## Presets

Algunos pares (F, k) tipicos:
- spots: F=0.035, k=0.062
- stripes: F=0.030, k=0.062
- solitons: F=0.030, k=0.055
- coral: F=0.0545, k=0.062
- maze: F=0.029, k=0.057


In [ ]:
PRESETS = {
    "spots": {"F": 0.035, "k": 0.062},
    "stripes": {"F": 0.030, "k": 0.062},
    "solitons": {"F": 0.030, "k": 0.055},
    "coral": {"F": 0.0545, "k": 0.062},
    "maze": {"F": 0.029, "k": 0.057},
}


def run_preset(name: str, steps: int = 400, show_every: int = 25):
    params = PRESETS[name]
    U, V = init_fields(n, seed_size=12, noise=0.02)
    F = params["F"]
    k = params["k"]
    for i in range(steps):
        U, V = gs_step(U, V, Du, Dv, F, k, dt)
        if i % show_every == 0:
            clear_output(wait=True)
            display(render_gray(V, size=400))
            print(f"preset={name} step={i} F={F} k={k}")
            time.sleep(0.01)

# Ejemplo: run_preset("stripes")


## Integracion con ascii_stream_engine

A continuacion se define un FrameSource sintetico que genera patrones
Gray-Scott. Cada read() avanza algunos pasos y devuelve un frame.

Tip: usa render_mode="raw" si queres ver el patron sin ASCII.


In [ ]:
from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    AsciiRenderer,
    FfmpegUdpOutput,
)


class ReactionDiffusionSource:
    def __init__(
        self,
        n: int = 200,
        steps_per_frame: int = 4,
        Du: float = 0.16,
        Dv: float = 0.08,
        F: float = 0.035,
        k: float = 0.065,
        dt: float = 1.0,
        seed_size: int = 12,
        noise: float = 0.02,
    ) -> None:
        self.n = n
        self.steps_per_frame = steps_per_frame
        self.Du = Du
        self.Dv = Dv
        self.F = F
        self.k = k
        self.dt = dt
        self.seed_size = seed_size
        self.noise = noise
        self.U = None
        self.V = None
        self._running = False

    def open(self) -> None:
        self.U, self.V = init_fields(self.n, self.seed_size, self.noise)
        self._running = True

    def read(self):
        if not self._running:
            return None
        for _ in range(self.steps_per_frame):
            self.U, self.V = gs_step(
                self.U, self.V, self.Du, self.Dv, self.F, self.k, self.dt
            )
        frame = (self.V * 255).astype(np.uint8)
        return frame

    def close(self) -> None:
        self._running = False


source = ReactionDiffusionSource(n=200, steps_per_frame=4)
config = EngineConfig(
    host="127.0.0.1",
    port=1234,
    fps=30,
    render_mode="raw",
    raw_width=400,
    raw_height=400,
)
engine = StreamEngine(
    source=source,
    renderer=AsciiRenderer(),
    sink=FfmpegUdpOutput(),
    config=config,
)

# Para iniciar/detener:
# engine.start()
# engine.stop()


VLC:
- Local: udp://@127.0.0.1:1234
- Multicast: udp://@239.0.0.1:1234


In [ ]:
def laplacian(x: np.ndarray) -> np.ndarray:
    return (
        -x
        + 0.2
        * (
            np.roll(x, 1, 0)
            + np.roll(x, -1, 0)
            + np.roll(x, 1, 1)
            + np.roll(x, -1, 1)
        )
        + 0.05
        * (
            np.roll(x, (1, 1), (0, 1))
            + np.roll(x, (1, -1), (0, 1))
            + np.roll(x, (-1, 1), (0, 1))
            + np.roll(x, (-1, -1), (0, 1))
        )
    )


def init_fields(n: int, seed_size: int = 10, noise: float = 0.0):
    U = np.ones((n, n), dtype=np.float32)
    V = np.zeros((n, n), dtype=np.float32)
    s = n // 2
    r = seed_size // 2
    U[s - r : s + r, s - r : s + r] = 0.0
    V[s - r : s + r, s - r : s + r] = 1.0
    if noise > 0.0:
        U += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        V += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        np.clip(U, 0.0, 1.0, out=U)
        np.clip(V, 0.0, 1.0, out=V)
    return U, V


def gs_step(U, V, Du: float, Dv: float, F: float, k: float, dt: float):
    lapU = laplacian(U)
    lapV = laplacian(V)
    uvv = U * V * V
    U += (Du * lapU - uvv + F * (1 - U)) * dt
    V += (Dv * lapV + uvv - (F + k) * V) * dt
    np.clip(U, 0.0, 1.0, out=U)
    np.clip(V, 0.0, 1.0, out=V)
    return U, V


def to_image(V: np.ndarray, scale: int = 3) -> Image.Image:
    img = (V * 255).astype(np.uint8)
    return Image.fromarray(img, mode="L").resize(
        (img.shape[1] * scale, img.shape[0] * scale)
    )


In [ ]:
def laplacian(x):
    return (
        -x
        + 0.2 * (
            np.roll(x, 1, 0)
            + np.roll(x, -1, 0)
            + np.roll(x, 1, 1)
            + np.roll(x, -1, 1)
        )
        + 0.05 * (
            np.roll(x, (1, 1), (0, 1))
            + np.roll(x, (1, -1), (0, 1))
            + np.roll(x, (-1, 1), (0, 1))
            + np.roll(x, (-1, -1), (0, 1))
        )
    )


def init_fields(n, seed_size=10, noise=0.0):
    U = np.ones((n, n), dtype=np.float32)
    V = np.zeros((n, n), dtype=np.float32)
    s = n // 2
    r = seed_size // 2
    U[s - r : s + r, s - r : s + r] = 0.0
    V[s - r : s + r, s - r : s + r] = 1.0
    if noise > 0.0:
        U += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        V += noise * (np.random.rand(n, n).astype(np.float32) - 0.5)
        np.clip(U, 0.0, 1.0, out=U)
        np.clip(V, 0.0, 1.0, out=V)
    return U, V


def gs_step(U, V, Du, Dv, F, k, dt):
    lapU = laplacian(U)
    lapV = laplacian(V)
    uvv = U * V * V
    U += (Du * lapU - uvv + F * (1 - U)) * dt
    V += (Dv * lapV + uvv - (F + k) * V) * dt
    np.clip(U, 0.0, 1.0, out=U)
    np.clip(V, 0.0, 1.0, out=V)
    return U, V


def to_image(V, scale=2):
    img = (V * 255).astype(np.uint8)
    pil = Image.fromarray(img, mode="L")
    if scale > 1:
        pil = pil.resize((pil.width * scale, pil.height * scale), Image.NEAREST)
    return pil
